# 06 & 07 - Entity Sentiment: Complete Solution

**Learning objective:** Design prompts that distinguish sentiment toward individual companies, then use token-level log probabilities and five independent samples to explore positivity and consistency.

This solution covers [06 - Entity Sentiment](../../content/06_entity_sentiment.ipynb) and [07 - Sentiment Uncertainty](../../content/07_sentiment_uncertainty.ipynb). Run the cells in order from a fresh Python kernel.

**Prerequisites:** Install `openai`, `pydantic` (version 2 or newer), and `tiktoken` in the notebook kernel (for example, `%pip install openai 'pydantic>=2' tiktoken`). Set `OPENAI_API_KEY` in the environment or replace the API-key placeholder below. The workshop's OpenAI Chat Completions API and Pydantic structured-output pattern are used throughout. `gpt-4.1` supports structured outputs, temperature sampling, and `logprobs` / `top_logprobs`.

The supplied article is fictional. The analysis concerns its portrayal of the named companies, not claims about their actual businesses.

In [ ]:
import os

API_KEY = os.environ.get("OPENAI_API_KEY", "insert key here")
MODEL = "gpt-4.1"  # Workshop model with logprobs and temperature support.

In [ ]:
import json
import math
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from statistics import fmean, stdev
from typing import Literal

import tiktoken
from openai import OpenAI
from pydantic import BaseModel, Field


def make_client():
    if not API_KEY or API_KEY == "insert key here":
        raise ValueError("Set OPENAI_API_KEY or replace the API_KEY placeholder.")
    return OpenAI(api_key=API_KEY, timeout=120.0, max_retries=0)


client = make_client()

## Load the Article

This cell is complete. It locates `data/text/article.txt` from the repository root, `content/`, or `data/bin/`; no path editing is needed. Read the article before evaluating either prompt.

In [ ]:
article_path = next(
    (parent / "data" / "text" / "article.txt"
     for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "text" / "article.txt").is_file()),
    None,
)
if article_path is None:
    raise FileNotFoundError(
        "Cannot find data/text/article.txt. Start the kernel inside this repository."
    )
doc_article = article_path.read_text(encoding="utf-8")
print(doc_article)

## Exercise 1: Entity-Level Sentiment Analysis

1. Identify all named companies, resolving aliases such as DHL / DHL Group.
2. Assess the article's sentiment toward each company separately. Do not transfer the headline's tone or another company's problems to every entity.
3. Choose exactly one of `positive`, `negative`, or `neutral`. Define how to handle mixed evidence and descriptive mentions.
4. Return one structured entry per company. The supplied Pydantic schema controls the format; the prompt controls the analysis.

In [ ]:
class EntitySentiment(BaseModel):
    entity: str = Field(description="Canonical name of a company in the article.")
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="The article's sentiment toward this company, not its overall tone."
    )


class NewsAnalysis(BaseModel):
    entity_sentiments: list[EntitySentiment] = Field(
        description="One entry for each named company, with no duplicate aliases."
    )

In [ ]:
ANALYSIS_PROMPT = """
You are a careful news-analysis assistant. Use only the supplied article.
Identify every named company, including brief descriptive mentions near the end.
Exclude people, places, public authorities, and the project itself. Merge aliases
under one recognizable company name; list each company once in first-mention order.

Evaluate sentiment expressed toward EACH company, not the article's overall tone,
the project's success, or your prior beliefs about the company. Weigh favorable
outcomes and praise against criticism, losses, and operational problems that
actually concern that company. Distinguish facts from untested speculation.
Choose exactly one label per company:
- positive: the balance of explicit evidence is clearly favorable;
- negative: the balance of explicit evidence is clearly unfavorable;
- neutral: descriptive only, insufficient evidence, or genuinely balanced signals.

Return the requested structured entity_sentiments list, with only entity and
sentiment in each entry. Do not invent companies or add commentary.
""".strip()

### Submit, Parse, and Display

Run this cell after editing the prompt. The SDK parses the structured response into `NewsAnalysis`; refusal, empty results, and duplicate names are checked explicitly.

In [ ]:
completion = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": ANALYSIS_PROMPT},
        {"role": "user", "content": f"Analyze this article:\n\n{doc_article}"},
    ],
    response_format=NewsAnalysis,
)
message = completion.choices[0].message
if message.refusal or message.parsed is None:
    raise RuntimeError(f"No parsed analysis returned: {message.refusal}")
analysis = message.parsed
names = [row.entity.strip().casefold() for row in analysis.entity_sentiments]
if not names or any(not name for name in names) or len(names) != len(set(names)):
    raise ValueError("Expected nonempty, unique company names. Refine the prompt.")
entity_sentiments = [row.model_dump() for row in analysis.entity_sentiments]
print(json.dumps(entity_sentiments, indent=2, ensure_ascii=False))

### Solution Discussion: Exercise 1

A reasonable reading is positive for Carrefour, Schneider Electric, and Siemens, and negative for DHL Group, Maersk, and Tesco. Microsoft and SAP have explicitly mixed evidence; Unilever's lost sales exceed the savings in the affected quarter. Procter & Gamble, Nestlé, and IBM are described without a clear evaluation. Amazon, Walmart, and Alibaba lack information needed to establish whether their arrangements are favorable.

These are reading guides, not hard-coded expected model labels. Review the actual list above. Neutral can mean either balanced evidence or insufficient evidence, two different situations that this three-label scheme combines.

**Reflect:** Which mixed cases changed when you altered the definition of neutral? Did the model miss brief company mentions or duplicate aliases? How would 'financial outcomes only' change the result compared with 'all favorable and unfavorable coverage'?

## Exercise 2: Repeated Analysis with Log Probabilities

1. Freeze the reviewed company inventory below before running the analysis. It contains the 15 companies in this article and makes this exercise independent of Exercise 1's kernel state.
2. Write a binary sentiment policy: only `positive` or `negative`. In particular, state how mixed or insufficient evidence is forced into two classes.
3. Run **exactly five independent iterations in parallel**. Each iteration samples an evidence assessment for every company, then requests one label token per company. No conversation history or results are shared across iterations.
4. Inspect the five scores per company and their mean and sample standard deviation.

**Why two stages?** Sampling a final label can change that label without changing its underlying probability distribution for a fixed input. Here `temperature=0.8` samples short evidence assessments, varying the context supplied to the scoring call. The one-token scoring call uses `temperature=1.0`. Score variation therefore reflects sensitivity to sampled evidence assessments as well as any serving nondeterminism. Five runs may still agree.

Each iteration uses one evidence request plus 15 scoring requests: **80 API requests** for one full five-run experiment. There are at most five concurrent requests. Automatic SDK retries are disabled; failures are surfaced rather than adding analysis runs.

In [ ]:
# Reviewed inventory for the supplied article; aliases are merged once, here.
ENTITY_NAMES = (
    "Carrefour", "DHL Group", "Microsoft", "Amazon",
    "Schneider Electric", "Siemens", "Maersk", "SAP", "Unilever",
    "Tesco", "Procter & Gamble", "Nestlé", "IBM", "Walmart", "Alibaba",
)
N_RUNS = 5
EVIDENCE_TEMPERATURE = 0.8
SCORE_TEMPERATURE = 1.0
print(json.dumps(ENTITY_NAMES, indent=2, ensure_ascii=False))


class EntityEvidence(BaseModel):
    entity: str = Field(description="Exact company name from the supplied inventory.")
    assessment: str = Field(
        description="Two short sentences weighing article evidence for this company."
    )


class EvidenceAnalysis(BaseModel):
    evidence: list[EntityEvidence] = Field(
        description="Exactly one evidence assessment for each supplied company."
    )

In [ ]:
BINARY_SENTIMENT_PROMPT = """
Assess the article's portrayal of the target company using only article evidence.
Choose positive when the balance of concrete outcomes and explicit evaluations
is favorable, and negative when it is unfavorable. Consider both commercial gains
and operational or financial costs. Do not spread one company's sentiment to others.

For mixed evidence, weigh the size and directness of the stated consequences,
giving realized outcomes more weight than speculative future interpretations.
If the evidence is exactly balanced or purely descriptive, use negative as an
explicit forced-choice tie-break, not as a claim that the article is critical.
Neutral is not an allowed label in this exercise.

A sampled evidence assessment is a reading aid, not new evidence; check it against
the article. Apply the same policy to every company.
""".strip()

### Make the Two Label Probabilities Identifiable

We use the exact lowercase tokens `positive` and `negative` as the two labels. The tokenizer check verifies that each is exactly one distinct token for `MODEL`. Every scoring response contains only that one token, so its position is unambiguous. An equal `logit_bias` of +100 encourages the two labels without favoring either relative to the other. Returned log probabilities are those of this biased, temperature-controlled scoring request.

For each company in each run, use the log probabilities at the **same label position**:

`positive_score = P(positive) / (P(positive) + P(negative))`

The code subtracts the larger log probability before exponentiating (a two-class log-sum-exp calculation). It requests the top 20 alternatives and requires both exact label tokens to be present with usable log probabilities. Missing alternatives, sentinel values, refusals, or unexpected tokens stop the experiment explicitly; none are treated as zero and no partial five-run summary is reported. The probability of the whole generated response is never used.

In [ ]:
LABEL_TO_SENTIMENT = {"positive": "positive", "negative": "negative"}
encoding = tiktoken.encoding_for_model(MODEL)
label_token_ids = {}
for label in LABEL_TO_SENTIMENT:
    token_ids = encoding.encode(label)
    if len(token_ids) != 1 or encoding.decode(token_ids) != label:
        raise ValueError(f"{label!r} is not one exact token for {MODEL}.")
    label_token_ids[label] = token_ids[0]
if len(set(label_token_ids.values())) != 2:
    raise ValueError("The two labels must have different token IDs.")
LABEL_BIAS = {str(token_id): 100 for token_id in label_token_ids.values()}
print({label: {"sentiment": LABEL_TO_SENTIMENT[label], "token_id": token_id}
       for label, token_id in label_token_ids.items()})


class ScoreUnavailableError(RuntimeError):
    pass


def extract_positive_score(choice):
    if choice.message.refusal:
        raise ScoreUnavailableError(f"Scoring refused: {choice.message.refusal}")
    token_infos = choice.logprobs.content if choice.logprobs else None
    if not token_infos or len(token_infos) != 1:
        raise ScoreUnavailableError("Expected log probabilities for one label token.")
    token_info = token_infos[0]
    if (choice.message.content not in LABEL_TO_SENTIMENT
            or token_info.token != choice.message.content):
        raise ScoreUnavailableError("Expected exactly positive or negative, with no whitespace.")

    # Use the sampled token too: it need not be repeated in top_logprobs.
    logps = {item.token: item.logprob for item in token_info.top_logprobs}
    logps[token_info.token] = token_info.logprob
    missing = set(LABEL_TO_SENTIMENT) - logps.keys()
    if missing:
        raise ScoreUnavailableError(
            f"Alternative-label log probability unavailable for {sorted(missing)}; "
            "no score computed. Review the prompt/model; do not substitute zero."
        )
    a, b = logps["positive"], logps["negative"]
    # The API can use -9999.0 for an unavailable log probability.
    if any(not math.isfinite(value) or value <= -9999.0 for value in (a, b)):
        raise ScoreUnavailableError("Non-finite or sentinel label log probability.")
    offset = max(a, b)
    positive_weight = math.exp(a - offset)
    negative_weight = math.exp(b - offset)
    score = positive_weight / (positive_weight + negative_weight)
    return score, LABEL_TO_SENTIMENT[token_info.token]

### One Independent Iteration

The first request samples an evidence assessment. Each subsequent request scores one company with the same article, policy, and that run's company-specific assessment. Requests are fresh conversations. The inventory is checked by exact name before scoring. `max_tokens=1` deliberately ends the scoring response after the label; a `length` finish reason is normal here.

In [ ]:
def run_iteration(run_id):
    with make_client() as run_client:
        evidence_completion = run_client.beta.chat.completions.parse(
            model=MODEL,
            temperature=EVIDENCE_TEMPERATURE,
            messages=[
                {"role": "system", "content": BINARY_SENTIMENT_PROMPT + (
                    "\nPrepare evidence assessments, not final labels. For each supplied "
                    "company, weigh favorable and unfavorable passages in two short "
                    "sentences. Note missing evidence. Preserve every supplied name "
                    "exactly once and add no companies."
                )},
                {"role": "user", "content": (
                    f"Companies: {json.dumps(ENTITY_NAMES, ensure_ascii=False)}\n\n"
                    f"Article:\n{doc_article}"
                )},
            ],
            response_format=EvidenceAnalysis,
        )
        message = evidence_completion.choices[0].message
        if message.refusal or message.parsed is None:
            raise RuntimeError(f"No parsed evidence returned: {message.refusal}")
        evidence_rows = message.parsed.evidence
        evidence_by_entity = {row.entity: row.assessment for row in evidence_rows}
        if (len(evidence_rows) != len(ENTITY_NAMES)
                or set(evidence_by_entity) != set(ENTITY_NAMES)
                or any(not text.strip() for text in evidence_by_entity.values())):
            raise ValueError("Evidence must cover the fixed inventory exactly once.")

        rows = []
        for entity in ENTITY_NAMES:
            score_completion = run_client.chat.completions.create(
                model=MODEL,
                temperature=SCORE_TEMPERATURE,
                max_tokens=1,
                logprobs=True,
                top_logprobs=20,
                logit_bias=LABEL_BIAS,
                messages=[
                    {"role": "system", "content": BINARY_SENTIMENT_PROMPT + (
                        "\nOutput exactly one lowercase word: positive or negative."
                        " No explanation, punctuation, whitespace, or neutral label."
                    )},
                    {"role": "user", "content": (
                        f"Target company: {entity}\n\nArticle:\n{doc_article}\n\n"
                        f"Sampled evidence assessment:\n{evidence_by_entity[entity]}"
                    )},
                ],
            )
            try:
                score, sentiment = extract_positive_score(score_completion.choices[0])
            except ScoreUnavailableError as exc:
                raise ScoreUnavailableError(f"Run {run_id}, {entity}: {exc}") from exc
            rows.append({
                "run": run_id, "entity": entity, "sentiment": sentiment,
                "positive_score": score, "assessment": evidence_by_entity[entity],
            })
        return rows

### Execute Exactly Five Runs in Parallel

Run this cell once per prompt experiment. The five workers each process the full inventory; companies are scored sequentially within each worker. Raw assessments and sampled labels are retained in `run_results` for inspection. Re-running this cell starts a new five-run experiment.

In [ ]:
run_results = []
with ThreadPoolExecutor(max_workers=N_RUNS) as executor:
    futures = {executor.submit(run_iteration, run_id): run_id
               for run_id in range(1, N_RUNS + 1)}
    for future in as_completed(futures):
        run_id = futures[future]
        try:
            run_results.extend(future.result())
        except Exception as exc:
            raise RuntimeError(
                f"Iteration {run_id} failed; no five-run summary is valid. {exc}"
            ) from exc
run_results.sort(key=lambda row: (row["run"], ENTITY_NAMES.index(row["entity"])))
print(f"Completed {N_RUNS} iterations for {len(ENTITY_NAMES)} companies.")

In [ ]:
expected_keys = {(run_id, entity) for run_id in range(1, N_RUNS + 1)
                 for entity in ENTITY_NAMES}
actual_keys = [(row["run"], row["entity"]) for row in run_results]
if len(actual_keys) != len(expected_keys) or set(actual_keys) != expected_keys:
    raise ValueError("Need exactly five complete, aligned runs; refusing partial data.")
if any(not math.isfinite(row["positive_score"])
       or not 0.0 <= row["positive_score"] <= 1.0 for row in run_results):
    raise ValueError("Every positivity score must be finite and between zero and one.")

summary = []
for entity in ENTITY_NAMES:
    scores = [row["positive_score"] for row in run_results if row["entity"] == entity]
    summary.append({
        "entity": entity,
        "mean_positive_score": fmean(scores),
        "positive_score_std": stdev(scores),  # Sample standard deviation: denominator n - 1.
    })

print("Five per-run positivity scores for each company:")
for entity in ENTITY_NAMES:
    scores = [row["positive_score"] for row in run_results if row["entity"] == entity]
    print(f"{entity}: " + ", ".join(f"{score:.6f}" for score in scores))
print("\nEntity-level summary (full precision retained in summary):")
print(json.dumps([
    {key: round(value, 6) if isinstance(value, float) else value
     for key, value in row.items()} for row in summary
], indent=2, ensure_ascii=False))

### Solution Discussion: Exercise 2

- A higher `mean_positive_score` indicates more positive sentiment **under this scoring method**. A larger `positive_score_std` indicates greater score variation across the five runs. `stdev` uses the sample definition, with denominator 4.
- Clear outcomes such as Carrefour's gains and DHL's penalties give useful comparison cases. Inspect Microsoft and SAP for competing evidence. Inspect the retained `assessment` values in `run_results` when their scores differ. Mixed cases need not have the largest observed standard deviation.
- The completed prompt deliberately maps balanced or absent evidence to negative. Thus a low score for IBM or Amazon may reflect the forced-choice policy rather than negative reporting. Try a different explicit tie-break policy and run a new five-run experiment to see its effect.
- In the live validation run, Carrefour, Schneider Electric, and Siemens scored near one; the other companies scored near zero under this prompt's negative tie-break. Variation was very small, with only a tiny visible change for Microsoft. This is one observed run, not an expected numeric answer. Six-decimal display rounding can hide small differences; `summary` and `run_results` retain full precision.
- Near-zero variation means only that these five sampled assessments led to similar probabilities. It does not establish that the extracted evidence or selected sentiment was correct. Identical probabilities can accompany different sampled final labels.
- These are **exploratory indicators of model confidence and consistency, not calibrated guarantees of correctness or reliability**. They depend on the prompt, two-token encoding, temperature, label bias, sampled assessments, model, and article. Five runs are a small sample, and binary forcing discards the neutral distinction.

**Reflect:** Which companies have conflicting evidence versus missing evidence? Does changing the tie-break rule shift the mean more than sampling does? How would you distinguish confident model agreement from a consistently biased prompt?